# Use cases

Two concrete lookups on top of the mock data:

1. **per compound** — all targets it has been tested against
2. **per target** — all compounds tested against it

This connects to `probe.db`, the on-disk database built from `staging/_template`
by `examples/build_mock_db.py`. Run that script first if the file does not exist
yet:

```bash
uv run python examples/build_mock_db.py
```

In [1]:
from pathlib import Path

import pandas as pd

from probedb import ProbeDB

pd.set_option("display.max_colwidth", 44)
pd.set_option("display.width", 170)

DB_PATH = Path("..") / "probe.db"
assert DB_PATH.exists(), f"{DB_PATH} not found -- run examples/build_mock_db.py first"

db = ProbeDB(DB_PATH, create=False)

db.counts()

,table,rows
0,compound,3
1,chembl,3
2,uniprot,8
3,target,6
4,target_uniprot,9
5,bioactivity_source,10
6,bioactivity_group,8
7,bioactivity,13


## Use case 1: per compound, all targets

`db.bioactivities(compound=...)` returns one row per measurement, so several
rows can point at the same target. `drop_duplicates` collapses that down to
the distinct targets a compound has been measured against.

In [2]:
def targets_for_compound(db, compound):
    hits = db.bioactivities(compound=compound)
    return hits[["target_type", "target"]].drop_duplicates().reset_index(drop=True)


for compound in db.table("compound")["name"]:
    targets = targets_for_compound(db, compound)
    plural = "" if len(targets) == 1 else "s"
    print(f"{compound} -- {len(targets)} target{plural}")
    for _, row in targets.iterrows():
        print(f"  [{row.target_type}] {row.target}")
    print()

BI-2536 -- 3 targets
  [protein] Serine/threonine-protein kinase PLK1
  [protein] Bromodomain-containing protein 4
  [complex] Cyclin-dependent kinase 1/cyclin B1

(+)-JQ1 -- 2 targets
  [protein] Bromodomain-containing protein 4
  [protein] Bromodomain-containing protein 2

Olaparib -- 3 targets
  [protein] Poly [ADP-ribose] polymerase 1
  [family] PARP 1, 2 and 3
  [protein] Serine/threonine-protein kinase PLK1



## Use case 2: per target, all compounds

Same idea in the other direction: `db.bioactivities(target=...)` accepts a
target id, a UniProt accession or an HGNC symbol. Iterating `db.table("target")`
walks proteins, complexes and families alike.

In [3]:
def compounds_for_target(db, target_id):
    hits = db.bioactivities(target=target_id)
    return hits[["compound"]].drop_duplicates().reset_index(drop=True)


for _, target in db.table("target").iterrows():
    compounds = compounds_for_target(db, target.target_id)
    plural = "" if len(compounds) == 1 else "s"
    name = target["name"]
    print(f"{name} ({target.type}) -- {len(compounds)} compound{plural}")
    for _, row in compounds.iterrows():
        print(f"  {row.compound}")
    print()

Serine/threonine-protein kinase PLK1 (protein) -- 2 compounds
  BI-2536
  Olaparib

Bromodomain-containing protein 4 (protein) -- 2 compounds
  BI-2536
  (+)-JQ1

Bromodomain-containing protein 2 (protein) -- 1 compound
  (+)-JQ1

Poly [ADP-ribose] polymerase 1 (protein) -- 1 compound
  Olaparib

PARP 1, 2 and 3 (family) -- 1 compound
  Olaparib

Cyclin-dependent kinase 1/cyclin B1 (complex) -- 1 compound
  BI-2536

